# Ultra-Lightweight and Interpretable Intrusion Detection in Software-Defined Networking: A Consensus XAI-Driven Feature Reduction Framework

Author: Mohammad Javad Akbari  
Objective: Implement a Zero-Leakage NIDS pipeline utilizing Dual-XAI (SHAP + LIME) to calculate a Jaccard Consensus and dynamic 95% variance elbow cutoff, thereby minimizing SDN controller-plane latency. 

## Abstract
Modern network architectures increasingly rely on Software-Defined Networking (SDN) to decouple the control logic from data forwarding planes. However, this centralization introduces severe security risks. Existing NIDS paradigms frequently deploy heavy Deep Learning (DL) architectures that induce high computational overhead and unacceptable latency. This notebook implements an ultra-lightweight, high-throughput, and fully interpretable NIDS pipeline. By leveraging a Dual-XAI (SHAP + LIME) consensus mechanism for dimensionality reduction, we systematically eliminate non-informative flow features and deploy classical machine learning classifiers directly optimized for resource-constrained SDN controller environments.

## References
[1] A. H. Janabi, T. Kanakis, M. Johnson, "Survey: Intrusion Detection System in Software-Defined Networking," IEEE Access, 2024. DOI: 10.1109/ACCESS.2024.3493384  
[2] M. S. Elsayed, N.-A. Le-Khac, A. D. Jurcut, "InSDN: A Novel SDN Intrusion Dataset," IEEE Access, 2020. DOI: 10.1109/ACCESS.2020.3022633  
[3] M. Tserenkhuu, M. D. Hossain, Y. Taenaka, Y. Kadobayashi, "Intrusion Detection System Framework for SDN-Based IoT Networks Using Deep Learning Approaches With XAI-Based Feature Selection Techniques and Domain-Constrained Features," IEEE Access, 2025. DOI: 10.1109/ACCESS.2025.3595595  
[4] M. T. Ribeiro, S. Singh, C. Guestrin, "Why Should I Trust You? Explaining the Predictions of Any Classifier," Proc. 22nd ACM SIGKDD, 2016. DOI: 10.1145/2939672.2939778  
[5] S. M. Lundberg, S.-I. Lee, "A Unified Approach to Interpreting Model Predictions," NeurIPS (NIPS), 2017.  
[6] I. Sharafaldin, A. H. Lashkari, A. A. Ghorbani, "Toward Generating a New Intrusion Detection Dataset and Intrusion Traffic Characterization (CSE-CIC-IDS2018)," ICISSP, 2018. DOI: 10.5220/0006639801080116  
[7] A. Mohamed, "Cleaned CSE-CIC-IDS2018 Dataset," Mendeley Data, V1, 2024. DOI: 10.17632/29hdbdzx2r.1  
[8] F. Pedregosa et al., "Scikit-learn: Machine Learning in Python," JMLR, vol. 12, 2011.  
[9] G. Ke et al., "LightGBM: A Highly Efficient Gradient Boosting Decision Tree," NeurIPS, vol. 30, 2017.  
[10] T. Chen, C. Guestrin, "XGBoost: A Scalable Tree Boosting System," Proc. 22nd ACM SIGKDD, 2016. DOI: 10.1145/2939672.2939785"

## 1. Setup & Global Configuration
Initialize the computational environment, declare deterministic seeds, and configure structural limits for memory safety.

In [ ]:
import os
import gc
import time
import psutil
import warnings
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.cluster import MiniBatchKMeans

import xgboost as xgb
import lightgbm as lgb
import shap
import lime.lime_tabular
import optuna
import joblib

# Attempt PyTorch import for CNN baseline; degrade gracefully if unavailable
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import DataLoader, TensorDataset
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False
    print("PyTorch not found. CNN baseline will be skipped.")

warnings.filterwarnings('ignore')

# Central Configuration Dictionary
CONFIG = {
    'INSDN_PATH': Path('./datasets/InSDN_Normal_and_Attack_Combined.csv'),
    'CIC_PATH': Path('./datasets/CSE-CIC-IDS2018/'),
    'ARTIFACTS_DIR': Path('./artifacts/'),
    'MODELS_DIR': Path('./artifacts/models/'),
    'FIGURES_DIR': Path('./artifacts/figures/'),
    'RANDOM_STATE': 42,
    'MAX_ROWS_PER_CLASS': 50000,
    'CHUNKSIZE': 100000,
    'MICRO_BATCH_SIZE': 16 # For realistic edge NIDS latency simulation
}

# Ensure structural directories exist
for path in [CONFIG['ARTIFACTS_DIR'], CONFIG['MODELS_DIR'], CONFIG['FIGURES_DIR']]:
    path.mkdir(parents=True, exist_ok=True)

# Enforce Strict Determinism Everywhere
os.environ['PYTHONHASHSEED'] = str(CONFIG['RANDOM_STATE'])
random.seed(CONFIG['RANDOM_STATE'])
np.random.seed(CONFIG['RANDOM_STATE'])
if TORCH_AVAILABLE:
    torch.manual_seed(CONFIG['RANDOM_STATE'])
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def get_safe_name(name):
    """Sanitizes model names for consistent file saving/loading."""
    return name.replace(' ', '_').replace('(', '').replace(')', '').replace('/', '_')

## 2. Data Ingestion & Leakage-Free Preprocessing (Phase A)
Robust, chunked data loading with immediate dtype downcasting to respect hardware constraints (16GB RAM limit). Adheres to a strict 70/15/15 train/val/test split.

In [ ]:
def memory_usage():
    """Returns current process memory usage in MB."""
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / (1024 ** 2)

def clean_and_downcast(df):
    """Removes identifiers, handles NaNs, and downcasts numerical types."""
    identifiers = ['Flow ID', 'Source IP', 'Src IP', 'Destination IP', 'Dst IP', 
                   'Source Port', 'Src Port', 'Destination Port', 'Dst Port', 
                   'Timestamp', 'Protocol']
    cols_to_drop = [c for c in identifiers if c in df.columns]
    df = df.drop(columns=cols_to_drop, errors='ignore')
    
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.fillna(0, inplace=True)
    
    for col in df.columns:
        if df[col].dtype == 'float64':
            df[col] = df[col].astype('float32')
        elif df[col].dtype == 'int64':
            df[col] = pd.to_numeric(df[col], downcast='integer')
    return df

def generate_synthetic_fallback():
    print("WARNING: Dataset paths not found. Generating synthetic empirical distribution.")
    from sklearn.datasets import make_classification
    X, y = make_classification(n_samples=20000, n_features=75, n_informative=20, n_redundant=10, 
                               n_classes=3, weights=[0.6, 0.3, 0.1], random_state=CONFIG['RANDOM_STATE'])
    feature_names = [f'Flow_Feature_{i}' for i in range(75)]
    df = pd.DataFrame(X, columns=feature_names)
    df['Label'] = np.where(y == 0, 'Normal', np.where(y == 1, 'Attack_A', 'Attack_B'))
    return df

def ingest_data():
    dfs = []
    files_processed = 0
    
    if CONFIG['INSDN_PATH'].exists():
        print("Loading InSDN...")
        df_insdn = pd.read_csv(CONFIG['INSDN_PATH'])
        dfs.append(clean_and_downcast(df_insdn))
        files_processed += 1
        
    if CONFIG['CIC_PATH'].exists() and CONFIG['CIC_PATH'].is_dir():
        print("Loading CSE-CIC-IDS2018...")
        for file in CONFIG['CIC_PATH'].glob('*.csv'):
            for chunk in pd.read_csv(file, chunksize=CONFIG['CHUNKSIZE']):
                dfs.append(clean_and_downcast(chunk))
                break # Memory safety: 1 chunk per file for limited environments
            files_processed += 1
            
    if files_processed == 0:
        return generate_synthetic_fallback()
        
    combined_df = pd.concat(dfs, ignore_index=True)
    del dfs
    gc.collect()
    
    if 'Label' in combined_df.columns:
        combined_df = combined_df.groupby('Label').apply(
            lambda x: x.sample(min(len(x), CONFIG['MAX_ROWS_PER_CLASS']), random_state=CONFIG['RANDOM_STATE'])
        ).reset_index(drop=True)
        
    return combined_df

print(f"Initial Memory: {memory_usage():.2f} MB")
data = ingest_data()
print(f"Post-Load Memory: {memory_usage():.2f} MB")

label_col = 'Label' if 'Label' in data.columns else data.columns[-1]
X = data.drop(columns=[label_col])
y = data[label_col]

le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Strict Zero-Leakage Splitting
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y_encoded, test_size=0.15, stratify=y_encoded, random_state=CONFIG['RANDOM_STATE'])
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.1765, stratify=y_train_val, random_state=CONFIG['RANDOM_STATE'])

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_val_scaled = pd.DataFrame(scaler.transform(X_val), columns=X_val.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

print(f"Train: {X_train_scaled.shape}, Val: {X_val_scaled.shape}, Test: {X_test_scaled.shape}")

## 3. Baseline Evaluation (Full Feature Set)
Compute empirical benchmarks using all raw features to map baseline Deep Learning performance versus Classical ML models.

In [ ]:
baseline_metrics = []
global_baseline_models = {} 

def train_baseline(name, model, X_t, y_t):
    print(f"Training {name} (Full Features)...")
    start_train = time.time()
    
    if name == '1D CNN (PyTorch)':
        X_tensor = torch.tensor(X_t.values, dtype=torch.float32)
        y_tensor = torch.tensor(y_t, dtype=torch.long)
        dataset = TensorDataset(X_tensor, y_tensor)
        loader = DataLoader(dataset, batch_size=256, shuffle=True)
        
        optimizer = optim.Adam(model.parameters(), lr=0.001)
        criterion = nn.CrossEntropyLoss()
        
        model.train()
        for epoch in range(5):
            for batch_X, batch_y in loader:
                optimizer.zero_grad()
                out = model(batch_X.unsqueeze(1))
                loss = criterion(out, batch_y)
                loss.backward()
                optimizer.step()
        torch.save(model.state_dict(), CONFIG['MODELS_DIR'] / f"{get_safe_name(name)}.pth")
    else:
        model.fit(X_t, y_t)
        joblib.dump(model, CONFIG['MODELS_DIR'] / f"{get_safe_name(name)}.pkl")
        
    train_time = time.time() - start_train
    global_baseline_models[name] = model
    return train_time

num_classes = len(np.unique(y_train))
models = {
    # Deepened MLP to avoid strawman comparison, but viable on R5 CPU
    'MLP (DL Baseline)': MLPClassifier(hidden_layer_sizes=(128, 64, 32), max_iter=30, early_stopping=True, random_state=CONFIG['RANDOM_STATE'])
}

if TORCH_AVAILABLE:
    class Deep1DCNN(nn.Module):
        def __init__(self, num_classes):
            super(Deep1DCNN, self).__init__()
            self.net = nn.Sequential(
                nn.Conv1d(1, 16, kernel_size=3, padding=1),
                nn.BatchNorm1d(16),
                nn.ReLU(),
                nn.MaxPool1d(2),
                nn.Conv1d(16, 32, kernel_size=3, padding=1),
                nn.BatchNorm1d(32),
                nn.ReLU(),
                nn.AdaptiveMaxPool1d(1), # Handles dynamic feature lengths seamlessly
                nn.Flatten(),
                nn.Linear(32, num_classes)
            )
        def forward(self, x):
            return self.net(x)
            
    models['1D CNN (PyTorch)'] = Deep1DCNN(num_classes)

for name, clf in models.items():
    train_time = train_baseline(name, clf, X_train_scaled, y_train)
    # Note: Accuracy/F1 evaluation moved to Phase 6 Micro-batching to ensure fairness
    gc.collect()

## 4. Dual-XAI Consensus & Dynamic Reduction (Phase B)
Train an ultra-fast proxy explainer (LightGBM) to execute global SHAP evaluation and local LIME aggregation. Utilize the Jaccard Similarity index for validation and the 95% Cumulative Variance Threshold to isolate the optimal subspace dimension.

In [ ]:
print("Training proxy LightGBM for XAI Interpretation...")
proxy_model = lgb.LGBMClassifier(n_estimators=50, random_state=CONFIG['RANDOM_STATE'], n_jobs=-1, verbosity=-1)
proxy_model.fit(X_train_scaled, y_train)

print("Executing Global SHAP extraction...")
explainer_shap = shap.TreeExplainer(proxy_model)
X_train_sample = shap.sample(X_train_scaled, 5000, random_state=CONFIG['RANDOM_STATE'])
shap_output = explainer_shap(X_train_sample)

shap_values = shap_output.values if hasattr(shap_output, "values") else shap_output
if len(shap_values.shape) == 3:
    shap_importance = np.abs(shap_values).mean(axis=(0, 2))
elif isinstance(shap_values, list):
    shap_importance = np.abs(np.array(shap_values)).mean(axis=(0, 1))
else:
    shap_importance = np.abs(shap_values).mean(axis=0)

shap_ranked = pd.Series(shap_importance, index=X_train_scaled.columns).sort_values(ascending=False)

print("Executing Local LIME Aggregation (K-Means Representative Sampling)...")
# MiniBatchKMeans is much faster on CPUs while guaranteeing statistical representation
kmeans = MiniBatchKMeans(n_clusters=200, random_state=CONFIG['RANDOM_STATE'], batch_size=1024)
kmeans.fit(X_train_scaled)
representative_samples = kmeans.cluster_centers_

explainer_lime = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train_scaled.values, 
    feature_names=X_train_scaled.columns.tolist(), 
    class_names=[str(c) for c in le.classes_],
    mode='classification',
    random_state=CONFIG['RANDOM_STATE']
)

lime_importance = {col: 0.0 for col in X_train_scaled.columns}
for i in range(len(representative_samples)):
    exp = explainer_lime.explain_instance(representative_samples[i], proxy_model.predict_proba, num_features=20)
    for feature, weight in exp.as_list():
        for col in X_train_scaled.columns:
            if col in feature:
                lime_importance[col] += abs(weight)
                break

lime_ranked = pd.Series(lime_importance).sort_values(ascending=False)

print("\n--- Jaccard Similarity & Consensus Filtering ---")
# Extract top 30 from both to form a consensus candidate pool
top_k_threshold = 30
top_shap = set(shap_ranked.head(top_k_threshold).index)
top_lime = set(lime_ranked.head(top_k_threshold).index)

intersection = top_shap.intersection(top_lime)
union = top_shap.union(top_lime)
jaccard_index = len(intersection) / len(union)
print(f"Jaccard Similarity (Top {top_k_threshold} Features): {jaccard_index:.4f}")

# The True Consensus: Only consider features agreed upon in the Union
candidate_features = list(union)
print(f"Consensus Candidate Pool: {len(candidate_features)} features.")

shap_norm = shap_importance / np.sum(shap_importance)
lime_weights = np.array([lime_importance[c] for c in X_train_scaled.columns])
lime_norm = lime_weights / (np.sum(lime_weights) + 1e-9)

# Filter combined weights strictly to the consensus candidates
consensus_importance = pd.Series((shap_norm + lime_norm) / 2, index=X_train_scaled.columns)
consensus_series = consensus_importance[candidate_features].sort_values(ascending=False)

# Normalize the consensus subset to 1.0 for the elbow method
consensus_series = consensus_series / consensus_series.sum()
cumulative_variance = consensus_series.cumsum()
optimal_k = (cumulative_variance >= 0.95).idxmax()
k_index = consensus_series.index.get_loc(optimal_k) + 1

print(f"Optimal Dimensions Retained for 95% Variance: {k_index} features (out of {len(X_train_scaled.columns)})")

selected_features = consensus_series.head(k_index).index.tolist()

X_train_red = X_train_scaled[selected_features]
X_val_red = X_val_scaled[selected_features]
X_test_red = X_test_scaled[selected_features]

## 5. Hyperparameter Tuning & Reduced Feature Training (Phase C)
Apply Bayesian Optimization (Optuna) exclusively over the lightweight algorithms (XGBoost, LightGBM) utilizing the structurally reduced dimensionality.

In [ ]:
def objective_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 150),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'random_state': CONFIG['RANDOM_STATE'],
        'n_jobs': -1
    }
    model = xgb.XGBClassifier(**params)
    model.fit(X_train_red, y_train)
    preds = model.predict(X_val_red)
    return f1_score(y_val, preds, average='macro')

def objective_lgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 150),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'random_state': CONFIG['RANDOM_STATE'],
        'n_jobs': -1
    }
    model = lgb.LGBMClassifier(**params, verbosity=-1)
    model.fit(X_train_red, y_train)
    preds = model.predict(X_val_red)
    return f1_score(y_val, preds, average='macro')

print("Starting Optuna Optimization...")
optuna.logging.set_verbosity(optuna.logging.WARNING)

study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(objective_xgb, n_trials=5) # Reduced trials for R5 constraints

study_lgb = optuna.create_study(direction='maximize')
study_lgb.optimize(objective_lgb, n_trials=5)

xgb_tuned = xgb.XGBClassifier(**study_xgb.best_params, random_state=CONFIG['RANDOM_STATE'], n_jobs=-1)
lgb_tuned = lgb.LGBMClassifier(**study_lgb.best_params, random_state=CONFIG['RANDOM_STATE'], n_jobs=-1, verbosity=-1)
rf_reduced = RandomForestClassifier(n_estimators=50, random_state=CONFIG['RANDOM_STATE'], n_jobs=-1)
dt_reduced = DecisionTreeClassifier(random_state=CONFIG['RANDOM_STATE'])

tuned_models = {
    'Decision Tree': dt_reduced,
    'Random Forest': rf_reduced,
    'XGBoost (Tuned)': xgb_tuned,
    'LightGBM (Tuned)': lgb_tuned
}

for name, model in tuned_models.items():
    model.fit(X_train_red, y_train)
    joblib.dump(model, CONFIG['MODELS_DIR'] / f"{get_safe_name(name)}_reduced.pkl")

print("Optimized reduced models successfully trained and serialized.")

## 6. Micro-Benchmarking & Streaming Simulation (Phase D)
Stress-test the NIDS utilizing the held-out test split, isolating throughput boundaries (Packets Per Second) and latency distributions to validate deployment viability on an SDN Edge Controller.

In [ ]:
test_metrics = []

def streaming_benchmark(name, model, X_test_df, y_test_arr, phase_name):
    """Simulates real-world Edge controller packet evaluation using micro-batches."""
    batch_size = CONFIG['MICRO_BATCH_SIZE']
    preds = []
    
    start_infer = time.perf_counter()
    if 'CNN' in name:
        model.eval()
        with torch.no_grad():
            for i in range(0, len(X_test_df), batch_size):
                batch = torch.tensor(X_test_df.iloc[i:i+batch_size].values, dtype=torch.float32).unsqueeze(1)
                out = model(batch)
                preds.extend(torch.argmax(out, dim=1).numpy())
    else:
        for i in range(0, len(X_test_df), batch_size):
            batch = X_test_df.iloc[i:i+batch_size]
            preds.extend(model.predict(batch))
            
    infer_time = time.perf_counter() - start_infer
    
    latency_us = (infer_time / len(X_test_df)) * 1e6
    pps = len(X_test_df) / infer_time
    acc = accuracy_score(y_test_arr, preds)
    macro_f1 = f1_score(y_test_arr, preds, average='macro')
    
    # Corrected Model Sizing Logic
    if phase_name == 'Full Baseline (Heavy)':
        ext = '.pth' if 'CNN' in name else '.pkl'
        model_path = CONFIG['MODELS_DIR'] / f"{get_safe_name(name)}{ext}"
    else:
        model_path = CONFIG['MODELS_DIR'] / f"{get_safe_name(name)}_reduced.pkl"
        
    disk_size_kb = os.path.getsize(model_path) / 1024 if model_path.exists() else 0
    
    test_metrics.append({
        'Model': name,
        'Phase': phase_name,
        'Latency/Sample (µs)': round(latency_us, 2),
        'Throughput (PPS)': int(pps),
        'Accuracy': round(acc, 4),
        'Macro-F1': round(macro_f1, 4),
        'Model Size (KB)': round(disk_size_kb, 1)
    })

print("Initiating strictly isolated micro-batch benchmarking against Test Split...")

for name, model in tuned_models.items():
    streaming_benchmark(name, model, X_test_red, y_test, 'Reduced (Consensus XAI)')

if 'MLP (DL Baseline)' in global_baseline_models:
    streaming_benchmark('MLP (DL Baseline)', global_baseline_models['MLP (DL Baseline)'], X_test_scaled, y_test, 'Full Baseline (Heavy)')
if '1D CNN (PyTorch)' in global_baseline_models:
    streaming_benchmark('1D CNN (PyTorch)', global_baseline_models['1D CNN (PyTorch)'], X_test_scaled, y_test, 'Full Baseline (Heavy)')

test_df = pd.DataFrame(test_metrics)
print("\n--- Final Model Performance & Benchmarks ---")
display(test_df)

# Visualization 1: Macro-F1 Bar Comparison
plt.figure(figsize=(10, 6))
sns.barplot(data=test_df, x='Model', y='Macro-F1', hue='Phase', palette='viridis')
plt.title('Performance Stability: Full Baseline vs. XAI-Reduced Subspace')
plt.ylim(0.0, 1.05)
plt.ylabel('Macro-F1 Score')
plt.xticks(rotation=15)
plt.savefig(CONFIG['FIGURES_DIR'] / 'f1_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

# Visualization 2: Latency vs. F1 Pareto Frontier Scatter Plot
plt.figure(figsize=(10, 6))
sns.scatterplot(data=test_df, x='Latency/Sample (µs)', y='Macro-F1', 
                hue='Model', style='Phase', s=250, palette='Set1')
plt.title('SDN Edge Viability Analysis: Processing Latency vs. Metric Integrity')
plt.grid(True, linestyle='--', alpha=0.7)
plt.savefig(CONFIG['FIGURES_DIR'] / 'latency_pareto.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nPipeline execution complete. All constraints satisfied.")